In [1]:
import requests
import pandas as pd
import time
from datetime import datetime, timezone

In [2]:

DATA_API = "https://data-api.polymarket.com"

In [4]:
all_trades = []

for offset in range(0, 10000, 100):
    response = requests.get(
        f"{DATA_API}/trades",
        params={
            "limit": 100,
            "offset": offset,
            "takerOnly": False
        },
        timeout=30
    )

    response.raise_for_status()

    batch = response.json()
    all_trades.extend(batch)

    print(f"Offset {offset}: {len(batch)} trades")

    if len(batch) < 100:
        break

    time.sleep(0.2)

trades_df = pd.DataFrame(all_trades)

print("Total trades:", len(trades_df))

Offset 0: 100 trades
Offset 100: 100 trades
Offset 200: 100 trades
Offset 300: 100 trades
Offset 400: 100 trades
Offset 500: 100 trades
Offset 600: 100 trades
Offset 700: 100 trades
Offset 800: 100 trades
Offset 900: 100 trades
Offset 1000: 100 trades
Offset 1100: 100 trades
Offset 1200: 100 trades
Offset 1300: 100 trades
Offset 1400: 100 trades
Offset 1500: 100 trades
Offset 1600: 100 trades
Offset 1700: 100 trades
Offset 1800: 100 trades
Offset 1900: 100 trades
Offset 2000: 100 trades
Offset 2100: 100 trades
Offset 2200: 100 trades
Offset 2300: 100 trades
Offset 2400: 100 trades
Offset 2500: 100 trades
Offset 2600: 100 trades
Offset 2700: 100 trades
Offset 2800: 100 trades
Offset 2900: 100 trades
Offset 3000: 100 trades
Offset 3100: 100 trades
Offset 3200: 100 trades
Offset 3300: 100 trades
Offset 3400: 100 trades
Offset 3500: 100 trades
Offset 3600: 100 trades
Offset 3700: 100 trades
Offset 3800: 100 trades
Offset 3900: 100 trades
Offset 4000: 100 trades
Offset 4100: 100 trades
Offs

SSLError: HTTPSConnectionPool(host='data-api.polymarket.com', port=443): Max retries exceeded with url: /trades?limit=100&offset=6900&takerOnly=False (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1016)')))

In [5]:
import requests
import pandas as pd
import time
import os

DATA_API = "https://data-api.polymarket.com"

session = requests.Session()

all_trades = []

start_offset = 6900
end_offset = 10000
batch_size = 100

for offset in range(start_offset, end_offset, batch_size):

    success = False

    for attempt in range(5):

        try:
            response = session.get(
                f"{DATA_API}/trades",
                params={
                    "limit": batch_size,
                    "offset": offset,
                    "takerOnly": False
                },
                timeout=30
            )

            response.raise_for_status()

            batch = response.json()
            success = True
            break

        except requests.exceptions.RequestException as e:

            print(
                f"Offset {offset} failed "
                f"(attempt {attempt + 1}/5): {e}"
            )

            if attempt < 4:
                wait_time = 2 ** attempt
                print(f"Retrying in {wait_time}s...")
                time.sleep(wait_time)

    if not success:
        print(f"Could not collect offset {offset}. Stopping.")
        break

    all_trades.extend(batch)

    print(
        f"Offset {offset}: "
        f"{len(batch)} trades | "
        f"Total: {len(all_trades)}"
    )

    # Save progress every 1,000 trades
    if len(all_trades) % 1000 == 0:

        trades_df = pd.DataFrame(all_trades)

        os.makedirs("../data/raw", exist_ok=True)

        trades_df.to_parquet(
            "../data/raw/trades_progress.parquet",
            index=False
        )

    time.sleep(0.5)

print("Collection finished.")
print("Total trades collected:", len(all_trades))

Offset 6900: 100 trades | Total: 100
Offset 7000: 100 trades | Total: 200
Offset 7100: 100 trades | Total: 300
Offset 7200: 100 trades | Total: 400
Offset 7300: 100 trades | Total: 500
Offset 7400: 100 trades | Total: 600
Offset 7500: 100 trades | Total: 700
Offset 7600: 100 trades | Total: 800
Offset 7700: 100 trades | Total: 900
Offset 7800: 100 trades | Total: 1000
Offset 7900: 100 trades | Total: 1100
Offset 8000: 100 trades | Total: 1200
Offset 8100: 100 trades | Total: 1300
Offset 8200: 100 trades | Total: 1400
Offset 8300: 100 trades | Total: 1500
Offset 8400: 100 trades | Total: 1600
Offset 8500: 100 trades | Total: 1700
Offset 8600: 100 trades | Total: 1800
Offset 8700: 100 trades | Total: 1900
Offset 8800: 100 trades | Total: 2000
Offset 8900: 100 trades | Total: 2100
Offset 9000: 100 trades | Total: 2200
Offset 9100: 100 trades | Total: 2300
Offset 9200: 100 trades | Total: 2400
Offset 9300: 100 trades | Total: 2500
Offset 9400: 100 trades | Total: 2600
Offset 9500: 100 trad

In [6]:
trades_df = pd.DataFrame(all_trades)

trades_df["datetime"] = pd.to_datetime(
    trades_df["timestamp"],
    unit="s",
    utc=True
)

print("Total trades:", len(trades_df))
print("Earliest:", trades_df["datetime"].min())
print("Latest:", trades_df["datetime"].max())

print(
    "Time span:",
    trades_df["datetime"].max() -
    trades_df["datetime"].min()
)

Total trades: 3100
Earliest: 2026-09-10 15:23:48+00:00
Latest: 2026-09-10 15:25:12+00:00
Time span: 0 days 00:01:24


In [ ]:
trades_df["datetime"] = pd.to_datetime(
    trades_df["timestamp"],
    unit="s",
    utc=True
)

print("Earliest:", trades_df["datetime"].min())
print("Latest:", trades_df["datetime"].max())

print(
    "Time span:",
    trades_df["datetime"].max() -
    trades_df["datetime"].min()
)